# 06 — Model Selection, Stability, and Explainability

This notebook focuses on judgement: how stable is the structure we discovered?


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_mutual_info_score
from sklearn.preprocessing import StandardScaler

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.data import make_customer_segmentation_data
from unsup_lab.evaluation import evaluate_k_range
from unsup_lab.reporting import cluster_profile


In [ ]:
dataset = make_customer_segmentation_data(n_customers=1_500, random_state=123)
features = dataset.features
x = StandardScaler().fit_transform(features)


## Internal metrics across k

In [ ]:
metrics = evaluate_k_range(
    x,
    estimator_factory=lambda k: KMeans(n_clusters=k, n_init=20, random_state=123),
    k_values=list(range(2, 11)),
)

metrics


## Seed stability

In [ ]:
labels_by_seed = []
for seed in range(20):
    model = KMeans(n_clusters=5, n_init=10, random_state=seed)
    labels_by_seed.append(model.fit_predict(x))

rows = []
for i in range(len(labels_by_seed)):
    for j in range(i + 1, len(labels_by_seed)):
        rows.append(
            {
                "seed_i": i,
                "seed_j": j,
                "adjusted_mutual_information": adjusted_mutual_info_score(
                    labels_by_seed[i],
                    labels_by_seed[j],
                ),
            }
        )

stability = pd.DataFrame(rows)
stability["adjusted_mutual_information"].describe()


## Explain the selected clustering

In [ ]:
selected = KMeans(n_clusters=5, n_init=20, random_state=123).fit_predict(x)
profile = cluster_profile(features, selected)

profile.round(2)


## Interpretation

A good unsupervised learning workflow should ask:

- Does the result change under another random seed?
- Does it change under another scaling method?
- Does it change when outliers are removed?
- Are the clusters actionable?
- Are the clusters stable enough to support decisions?

These questions are often more important than the choice of algorithm.
